# Problema 3 · Separación de frutas que se tocan o se superponen
**Tarea: funcionamiento de YOLO · Tópicos de Inteligencia Artificial**

**Problema:** identificar cada fruta y delimitar su contorno aunque varias estén juntas.
**Capacidad:** segmentación de instancias (una máscara independiente por fruta).

| Elemento | Lo que usamos |
|---|---|
| Versión | **YOLO11** de Ultralytics (la versión exacta de la librería se imprime en la sección 0) |
| Variante / tarea | `yolo11n-seg`: *nano*, segmentación de instancias |
| Arquitectura | **Backbone** CSP (Conv + C3k2 + SPPF + C2PSA) → **Neck** PAN-FPN (Upsample + Concat + C3k2) → **Cabeza** `Segment` |
| Cabeza de predicción | `Segment` = cabeza de detección desacoplada (caja con DFL + clases) **+** 32 coeficientes de máscara por candidato **+** rama `Proto` que genera 32 prototipos de máscara |
| Pesos | Preentrenados en COCO (`yolo11n-seg.pt`) y **ajustados (fine-tuning)** con el dataset de frutas |
| Dataset | *Fruit instance segmentation dataset for YOLOvFIS network* (Figshare, 6 clases, formato YOLOv8-seg) |

**Recorrido:** se sigue **una misma imagen de prueba** (la que tiene más frutas que se tocan) por las 7 etapas.
Las figuras dicen **[EVIDENCIA]** si salen del modelo ejecutado (tensores reales capturados con *hooks*) o
**[DIAGRAMA]** si son explicativas.

> En Colab: `Entorno de ejecución → Cambiar tipo de entorno → GPU T4` y luego ejecuta las celdas en orden.

## 0. Instalación y parámetros

In [ ]:
# @title 0.1 Instalar librerías
!pip install -q ultralytics

In [ ]:
# @title 0.2 Importaciones y parámetros { display-mode: "form" }
import os, time, json, random, shutil, zipfile, platform
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
import yaml
import ultralytics
from ultralytics import YOLO
from PIL import Image

# @markdown **Entrenamiento**
PESOS_BASE = "yolo11n-seg.pt"  # @param ["yolo11n-seg.pt", "yolo11s-seg.pt", "yolov8n-seg.pt"]
EPOCAS = 30  # @param {type:"integer"}
IMGSZ = 640  # @param {type:"integer"}
BATCH = 16  # @param {type:"integer"}
FRACCION = 1.0  # @param {type:"number"}
# @markdown `FRACCION` < 1 entrena con una parte del conjunto de entrenamiento (para pruebas rápidas).
PACIENCIA = 10  # @param {type:"integer"}
# @markdown **Posprocesamiento**
CONF = 0.25  # @param {type:"number"}
IOU_NMS = 0.7  # @param {type:"number"}
UMBRAL_MASCARA = 0.5  # @param {type:"number"}

RES = Path("resultados_frutas")
RES.mkdir(exist_ok=True)
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0); random.seed(0)


def guardar(fig, nombre):
    fig.savefig(RES / nombre, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def color(k):
    return plt.cm.tab20(k % 20)[:3]


print(f"Python {platform.python_version()} · torch {torch.__version__} · ultralytics {ultralytics.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay (será lento)")

## 1. Entrada y preparación
### 1.1 Obtener el dataset
Por defecto se descarga desde Figshare con su API (artículo 28436618). Si la descarga falla, baja el zip desde
la [página del dataset](https://figshare.com/articles/dataset/_b_Fruit_instance_segmentation_dataset_for_b_b_YOLOvFIS_network_b_/28436618)
y elige **Subir zip** o **Google Drive**.

In [ ]:
# @title 1.1 Descargar / cargar el dataset { display-mode: "form" }
FUENTE = "Figshare (automático)"  # @param ["Figshare (automático)", "Subir zip", "Google Drive"]
RUTA_ZIP_DRIVE = "/content/drive/MyDrive/frutas_yolovfis.zip"  # @param {type:"string"}
FIGSHARE_ID = 28436618
DATOS = Path("datos_frutas")
DESCARGAS = Path("descargas")
DESCARGAS.mkdir(exist_ok=True)


def extraer(zip_path):
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"{zip_path} no es un zip válido (quizá Figshare devolvió una página web). "
                           "Descárgalo a mano y usa 'Subir zip' o 'Google Drive'.")
    print(f"Descomprimiendo {Path(zip_path).name}...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATOS)
    # Zips dentro del zip (a veces Figshare empaqueta así)
    for interno in list(DATOS.rglob("*.zip")):
        with zipfile.ZipFile(interno) as z:
            z.extractall(interno.parent / interno.stem)
        interno.unlink()


def descargar_figshare():
    import requests
    cab = {"User-Agent": "Mozilla/5.0 (tarea-yolo-colab)"}
    info = requests.get(f"https://api.figshare.com/v2/articles/{FIGSHARE_ID}", headers=cab, timeout=60)
    info.raise_for_status()
    archivos = info.json()["files"]
    for f in archivos:
        print(f"  {f['name']}: {f['size'] / 1e6:.0f} MB")
    for f in archivos:
        destino = DESCARGAS / f["name"]
        if not destino.exists():
            with requests.get(f["download_url"], headers=cab, stream=True, timeout=120) as r:
                r.raise_for_status()
                total, leido = int(r.headers.get("Content-Length", 0)), 0
                with open(destino, "wb") as fh:
                    for bloque in r.iter_content(1 << 20):
                        fh.write(bloque)
                        leido += len(bloque)
                        if total:
                            print(f"\r  {f['name']}: {100 * leido / total:.0f}%", end="")
                print()
        if destino.suffix.lower() == ".zip":
            extraer(destino)


if not any(DATOS.rglob("*.txt")):
    DATOS.mkdir(exist_ok=True)
    if FUENTE.startswith("Figshare"):
        descargar_figshare()
    elif FUENTE == "Subir zip":
        from google.colab import files
        for nombre in files.upload():
            extraer(nombre)
    else:
        if EN_COLAB and RUTA_ZIP_DRIVE.startswith("/content/drive"):
            from google.colab import drive
            drive.mount("/content/drive")
        extraer(RUTA_ZIP_DRIVE)
print("Dataset en", DATOS.resolve())

In [ ]:
# @title 1.2 Localizar particiones y escribir frutas.yaml
ALIAS = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}


def etiqueta_de(img_path):
    # Misma regla que Ultralytics: .../images/x.jpg -> .../labels/x.txt
    p = str(img_path)
    i = p.rfind(f"{os.sep}images{os.sep}")
    return Path(p[:i] + f"{os.sep}labels{os.sep}" + p[i + 8:]).with_suffix(".txt")


SPLITS = {}
for d in sorted(DATOS.rglob("images")):
    if d.is_dir() and (d.parent / "labels").is_dir() and d.parent.name.lower() in ALIAS:   # train/images
        SPLITS.setdefault(ALIAS[d.parent.name.lower()], d)
    for sub in (d.iterdir() if d.is_dir() else []):                                          # images/train
        if sub.is_dir() and sub.name.lower() in ALIAS:
            SPLITS.setdefault(ALIAS[sub.name.lower()], sub)
assert "train" in SPLITS and "val" in SPLITS, f"No encontré train/valid en {DATOS}: {SPLITS}"
SPLITS.setdefault("test", SPLITS["val"])

NOMBRES = None
for y in DATOS.rglob("*.yaml"):
    cfg = yaml.safe_load(open(y)) or {}
    if "names" in cfg:
        n = cfg["names"]
        NOMBRES = {i: v for i, v in enumerate(n)} if isinstance(n, list) else {int(k): v for k, v in n.items()}
        print("Clases tomadas de", y)
        break
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGENES = {s: sorted(p for p in d.iterdir() if p.suffix.lower() in EXT) for s, d in SPLITS.items()}
if NOMBRES is None:
    ids = {int(l.split()[0]) for s in IMAGENES for p in IMAGENES[s] if etiqueta_de(p).exists()
           for l in etiqueta_de(p).read_text().splitlines() if l.strip()}
    NOMBRES = {i: f"clase_{i}" for i in range(max(ids) + 1)}

YAML = Path("frutas.yaml").resolve()
yaml.safe_dump({"path": str(DATOS.resolve()), "train": str(SPLITS["train"].resolve()),
                "val": str(SPLITS["val"].resolve()), "test": str(SPLITS["test"].resolve()),
                "names": NOMBRES}, open(YAML, "w"), allow_unicode=True, sort_keys=False)
NC = len(NOMBRES)
print(open(YAML).read())

### 1.3 Anotaciones, partición y estadísticas
Cada imagen tiene un `.txt` con **una línea por fruta**: `clase x1 y1 x2 y2 … xn yn` (polígono del contorno,
coordenadas normalizadas 0-1). Dos frutas de la misma clase son **dos líneas distintas**: eso es lo que hace que
sean instancias separadas.

El dataset trae 3 versiones aumentadas (rotaciones, inclinación) de cada foto. Roboflow las nombra
`<original>.rf.<hash>.jpg`, así que contamos fotos originales y revisamos que **la misma foto no aparezca en dos
particiones** (eso inflaría las métricas).

In [ ]:
# @title 1.3 Estadísticas del dataset
def leer_etiquetas(img_path, w, h):
    # -> lista de (clase, polígono Nx2 en píxeles)
    salida = []
    t = etiqueta_de(img_path)
    if not t.exists():
        return salida
    for linea in t.read_text().splitlines():
        v = linea.split()
        if len(v) < 5:
            continue
        c, xy = int(v[0]), np.array(v[1:], dtype=float)
        if len(xy) == 4:  # línea de caja (cx cy w h): se convierte a rectángulo
            cx, cy, bw, bh = xy
            xy = np.array([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2, cx - bw / 2, cy + bh / 2])
        salida.append((c, xy.reshape(-1, 2) * [w, h]))
    return salida


def original_de(p):
    return p.name.split(".rf.")[0]


filas, conteo, tam = [], {}, {}
for s, ims in IMAGENES.items():
    c = Counter()
    for p in ims:
        t = etiqueta_de(p)
        if t.exists():
            c.update(int(l.split()[0]) for l in t.read_text().splitlines() if l.strip())
    conteo[s] = c
    tam[s] = Counter(Image.open(p).size for p in random.Random(0).sample(ims, min(300, len(ims))))
    filas.append((s, len(ims), len({original_de(p) for p in ims}), sum(c.values())))

print(f"{'partición':<8}{'imágenes':>10}{'fotos originales':>18}{'frutas':>9}   " + "  ".join(f"{NOMBRES[i][:10]:>10}" for i in range(NC)))
for s, n, u, f in filas:
    print(f"{s:<8}{n:>10}{u:>18}{f:>9}   " + "  ".join(f"{conteo[s][i]:>10}" for i in range(NC)))
for s in IMAGENES:
    print(f"Tamaños más comunes en {s}: {tam[s].most_common(3)}")

orig = {s: {original_de(p) for p in ims} for s, ims in IMAGENES.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    if SPLITS[a] != SPLITS[b]:
        print(f"Fotos originales compartidas entre {a} y {b}: {len(orig[a] & orig[b])}")

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(NC)
for k, s in enumerate(IMAGENES):
    ax.bar(x + (k - 1) * 0.27, [conteo[s][i] for i in range(NC)], 0.27, label=f"{s} ({len(IMAGENES[s])} imágenes)")
ax.set_xticks(x, [NOMBRES[i] for i in range(NC)], rotation=20)
ax.set_ylabel("frutas anotadas (instancias)")
ax.set_title("[EVIDENCIA] Instancias por clase y partición")
ax.legend()
guardar(fig, "1_distribucion_clases.png")

In [ ]:
# @title 1.4 Elegir la imagen de seguimiento { display-mode: "form" }
# @markdown Vacío = la imagen de prueba con más pares de frutas que se tocan. O escribe un nombre de archivo de test.
IMAGEN_SEGUIMIENTO = ""  # @param {type:"string"}


def mascaras_gt(anns, w, h):
    m = np.zeros((len(anns), h, w), np.uint8)
    for k, (_, poly) in enumerate(anns):
        cv2.fillPoly(m[k], [np.round(poly).astype(np.int32)], 1)
    return m.astype(bool)


def pares_que_se_tocan(masks, dil=3):
    kernel = np.ones((2 * dil + 1, 2 * dil + 1), np.uint8)
    d = [cv2.dilate(m.astype(np.uint8), kernel).astype(bool) for m in masks]
    return [(i, j) for i in range(len(masks)) for j in range(i + 1, len(masks)) if (d[i] & masks[j]).any()]


def puntaje(p, escala=320):
    w, h = Image.open(p).size
    r = escala / max(w, h)
    anns = [(c, poly * r) for c, poly in leer_etiquetas(p, w, h)]
    if len(anns) < 2:
        return 0, len(anns)
    return len(pares_que_se_tocan(mascaras_gt(anns, max(1, round(w * r)), max(1, round(h * r))), 1)), len(anns)


if IMAGEN_SEGUIMIENTO:
    IMG_SEG = next(p for p in IMAGENES["test"] if p.name == IMAGEN_SEGUIMIENTO)
else:
    IMG_SEG = max(IMAGENES["test"], key=puntaje)
img0 = cv2.imread(str(IMG_SEG))            # BGR, como lo lee Ultralytics
H0, W0 = img0.shape[:2]
GT = leer_etiquetas(IMG_SEG, W0, H0)
GT_MASK = mascaras_gt(GT, W0, H0)
PARES_GT = pares_que_se_tocan(GT_MASK)
print(f"Imagen de seguimiento: {IMG_SEG.name}")
print(f"Dimensiones originales: {W0} x {H0} px (ancho x alto), 3 canales")
print(f"Frutas anotadas: {len(GT)} · pares que se tocan o se superponen: {len(PARES_GT)}")
print("Primeras líneas del archivo de etiquetas (formato YOLO-seg):")
for l in etiqueta_de(IMG_SEG).read_text().splitlines()[:3]:
    v = l.split()
    print(f"   clase={v[0]}  {len(v[1:]) // 2} vértices: {' '.join(v[1:7])} ...")

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(img0[..., ::-1]); axs[0].set_title(f"[EVIDENCIA] Original {W0}x{H0}")
axs[1].imshow(img0[..., ::-1])
for k, (c, poly) in enumerate(GT):
    axs[1].add_patch(plt.Polygon(poly, fc=(*color(k), 0.45), ec=color(k), lw=2))
    axs[1].text(*poly.min(0), f"{k}:{NOMBRES[c]}", color="white", fontsize=8, bbox=dict(fc=color(k), ec="none", pad=1))
axs[1].set_title(f"[EVIDENCIA] Anotación: {len(GT)} instancias (un color por fruta)")
for a in axs: a.axis("off")
guardar(fig, "1_imagen_seguimiento_anotada.png")

## Entrenamiento (fine-tuning)
Se parte de los pesos de COCO y se ajusta con las 6 clases de frutas. Con GPU T4 y 30 épocas tarda
aproximadamente 1 h. Si ya entrenaste, escribe la ruta de tu `best.pt` y se salta el entrenamiento.

In [ ]:
# @title Entrenar o cargar pesos { display-mode: "form" }
PESOS_ENTRENADOS = ""  # @param {type:"string"}
# @markdown Ejemplo: `/content/drive/MyDrive/yolo_frutas/best.pt`. Vacío = entrenar ahora.
GUARDAR_EN_DRIVE = False  # @param {type:"boolean"}

if PESOS_ENTRENADOS and Path(PESOS_ENTRENADOS).exists():
    MEJOR = Path(PESOS_ENTRENADOS)
else:
    if PESOS_ENTRENADOS.startswith("/content/drive") and EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    modelo_entr = YOLO(PESOS_BASE)
    modelo_entr.train(data=str(YAML), epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, fraction=FRACCION,
                      patience=PACIENCIA, device=DISPOSITIVO, project=str(Path("runs_frutas").resolve()),
                      name="seg", exist_ok=True, seed=0, plots=True)
    MEJOR = Path(modelo_entr.trainer.best)
    curvas = Path(modelo_entr.trainer.save_dir) / "results.png"
    if curvas.exists():
        fig = plt.figure(figsize=(16, 8)); plt.imshow(plt.imread(curvas)); plt.axis("off")
        plt.title("[EVIDENCIA] Curvas de entrenamiento (pérdidas de caja, máscara, clase y DFL; métricas en val)")
        guardar(fig, "entrenamiento_curvas.png")
    if GUARDAR_EN_DRIVE and EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/yolo_frutas", exist_ok=True)
        shutil.copy(MEJOR, "/content/drive/MyDrive/yolo_frutas/best.pt")
        print("Copiado a /content/drive/MyDrive/yolo_frutas/best.pt")
print("Pesos:", MEJOR)

In [ ]:
# @title Cargar el modelo entrenado e identificar la arquitectura
modelo = YOLO(str(MEJOR))
red = modelo.model.to("cuda" if torch.cuda.is_available() else "cpu").float().eval()
DEV = next(red.parameters()).device
cabeza = red.model[-1]
NOMBRES_MODELO = red.names
print(f"Tarea: {modelo.task} · capas: {len(red.model)} · parámetros: {sum(p.numel() for p in red.parameters()) / 1e6:.2f} M")
print(f"Cabeza: {type(cabeza).__name__} (hereda de {type(cabeza).__mro__[1].__name__}) · clases nc={cabeza.nc} · "
      f"coeficientes de máscara nm={cabeza.nm} · bins DFL reg_max={cabeza.reg_max} · strides={cabeza.stride.tolist()}")
print("Clases:", NOMBRES_MODELO)

## 2. Preprocesamiento
Se reproduce a mano lo que hace Ultralytics al predecir (`LetterBox`):

1. **Redimensionar** conservando la proporción: $r = \min(640/H_0,\ 640/W_0)$.
2. **Relleno (padding)** gris (114) centrado. En predicción se rellena solo hasta el siguiente múltiplo de 32
   (stride máximo), no hasta un cuadrado completo.
3. **BGR → RGB**, **HWC → CHW**, **normalización** a [0, 1] (÷255) y dimensión de lote → tensor `(1, 3, H, W)`.

**Recuperación de coordenadas originales:** $x_0 = (x - \text{pad}_x)/r$, $y_0 = (y - \text{pad}_y)/r$.

In [ ]:
# @title 2.1 Letterbox manual, tensor de entrada y comparación con Ultralytics
def letterbox(img, tam=IMGSZ, stride=32, minimo=True):
    h0, w0 = img.shape[:2]
    r = min(tam / h0, tam / w0)
    nw, nh = int(round(w0 * r)), int(round(h0 * r))
    dw, dh = tam - nw, tam - nh
    if minimo:                                   # relleno mínimo (múltiplo de 32), como en predict
        dw, dh = dw % stride, dh % stride
    dw, dh = dw / 2, dh / 2
    redim = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR) if (nw, nh) != (w0, h0) else img.copy()
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    lb = cv2.copyMakeBorder(redim, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return lb, redim, r, (left, top)


def a_tensor(lb):
    t = torch.from_numpy(np.ascontiguousarray(lb[..., ::-1].transpose(2, 0, 1)))  # BGR->RGB, HWC->CHW
    return (t.float() / 255.0).unsqueeze(0)                                        # [0,255] -> [0,1], lote


LB, REDIM, R, (PADX, PADY) = letterbox(img0)
X = a_tensor(LB).to(DEV)
print(f"Original: {W0}x{H0} -> r = {R:.4f} -> redimensionada {REDIM.shape[1]}x{REDIM.shape[0]} "
      f"-> con relleno {LB.shape[1]}x{LB.shape[0]} (pad izq={PADX}, arriba={PADY})")
print(f"Tensor de entrada: forma {tuple(X.shape)}, tipo {X.dtype}, rango [{X.min():.3f}, {X.max():.3f}], "
      f"media por canal RGB {X.mean((0, 2, 3)).cpu().numpy().round(3)}")

# Evidencia: el tensor que Ultralytics mete realmente a la red (pre-hook)
capturado = {}
h = red.register_forward_pre_hook(lambda m, inp: capturado.__setitem__("x", inp[0].detach().float().cpu()))
res_ultra = modelo.predict(str(IMG_SEG), imgsz=IMGSZ, conf=CONF, iou=IOU_NMS, verbose=False)[0]
h.remove()
xu = capturado["x"]
print(f"Tensor capturado dentro de Ultralytics: {tuple(xu.shape)}")
if xu.shape == X.shape:
    print(f"Diferencia máxima con nuestro preprocesamiento manual: {(xu - X.cpu()).abs().max():.2e}")

# Recuperación de coordenadas: un vértice anotado -> letterbox -> original
c0, poly0 = GT[0]
x_o, y_o = poly0[0]
x_lb, y_lb = x_o * R + PADX, y_o * R + PADY
print(f"Vértice de la fruta 0: original ({x_o:.1f}, {y_o:.1f}) -> en el tensor ({x_lb:.1f}, {y_lb:.1f}) "
      f"-> recuperado ({(x_lb - PADX) / R:.1f}, {(y_lb - PADY) / R:.1f})")

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
axs[0].imshow(img0[..., ::-1]); axs[0].set_title(f"Original {W0}x{H0}")
axs[1].imshow(REDIM[..., ::-1]); axs[1].set_title(f"Redimensionada x{R:.3f}: {REDIM.shape[1]}x{REDIM.shape[0]}")
axs[2].imshow(LB[..., ::-1]); axs[2].set_title(f"Con relleno: {LB.shape[1]}x{LB.shape[0]}")
axs[2].add_patch(plt.Rectangle((PADX, PADY), REDIM.shape[1], REDIM.shape[0], fill=False, ec="red", lw=2))
axs[2].plot(x_lb, y_lb, "y*", ms=15)
for k, nom in enumerate("RGB"):
    axs[3].hist(X[0, k].flatten().cpu().numpy(), bins=50, alpha=0.5, color=nom.lower(), label=nom)
axs[3].set_title("Tensor normalizado: valores por canal"); axs[3].legend()
for a in axs[:3]: a.axis("off")
fig.suptitle("[EVIDENCIA] Preprocesamiento de la imagen de seguimiento")
guardar(fig, "2_preprocesamiento.png")

## 3. Backbone: extracción de características
Se registra un *forward hook* en **cada capa** de la red y se ejecuta con el tensor de la sección 2. Así las formas y
activaciones son las reales del modelo. La columna `de` dice de qué capa recibe la entrada (-1 = la anterior).

In [ ]:
# @title 3.1 Hooks en todas las capas: tabla de formas reales
ACT, ENTRADAS = {}, {}


def hook(k):
    def f(m, inp, out):
        ACT[k] = out
        ENTRADAS[k] = inp[0]
    return f


hooks = [capa.register_forward_hook(hook(k)) for k, capa in enumerate(red.model)]
with torch.no_grad():
    t0 = time.perf_counter()
    SALIDA = red(X)
    T_INFER = time.perf_counter() - t0
for hk in hooks:
    hk.remove()


def forma(o):
    if torch.is_tensor(o):
        return tuple(o.shape)
    if isinstance(o, dict):
        return {k: forma(v) for k, v in o.items() if not isinstance(v, (list, dict))}
    if isinstance(o, (list, tuple)):
        return [forma(v) for v in o]
    return type(o).__name__


print(f"{'capa':>4} {'tipo':<9} {'de':<14} {'forma de salida':<26} {'stride':>6} {'parámetros':>11}")
for k, capa in enumerate(red.model):
    o = ACT[k]
    s = f"{X.shape[-1] // o.shape[-1]}" if torch.is_tensor(o) and o.dim() == 4 else ""
    zona = "backbone" if k <= 10 else ("neck" if k < len(red.model) - 1 else "cabeza")
    print(f"{k:>4} {type(capa).__name__:<9} {str(capa.f):<14} {str(forma(o))[:60]:<26} {s:>6} "
          f"{sum(p.numel() for p in capa.parameters()):>11,}  {zona}")

In [ ]:
# @title 3.2 Activaciones de varias capas del backbone
CAPAS_BB = [0, 2, 4, 6, 8, 10]
fig, axs = plt.subplots(len(CAPAS_BB), 9, figsize=(20, 2.4 * len(CAPAS_BB)))
for fila, k in enumerate(CAPAS_BB):
    a = ACT[k][0].float().cpu()
    top = a.mean((1, 2)).argsort(descending=True)[:8]
    axs[fila, 0].imshow(LB[..., ::-1]); axs[fila, 0].imshow(cv2.resize(a.mean(0).numpy(), (LB.shape[1], LB.shape[0])), alpha=0.6, cmap="jet")
    axs[fila, 0].set_title(f"capa {k} {type(red.model[k]).__name__}\n{tuple(a.shape)} (media)", fontsize=8)
    for j, ch in enumerate(top):
        axs[fila, j + 1].imshow(a[ch], cmap="viridis")
        axs[fila, j + 1].set_title(f"canal {ch.item()}", fontsize=7)
for a in axs.flat: a.axis("off")
fig.tight_layout()
fig.suptitle("[EVIDENCIA] Activaciones reales (hooks): 8 canales con mayor activación media por capa", y=1.01)
guardar(fig, "3_backbone_activaciones.png")

# Kernels aprendidos de la primera convolución
w = red.model[0].conv.weight.detach().float().cpu()          # (salidas, 3, k, k)
fig, axs = plt.subplots(2, w.shape[0] // 2, figsize=(16, 2.6))
for j, a in enumerate(axs.flat):
    k_ = w[j].permute(1, 2, 0).numpy()
    a.imshow((k_ - k_.min()) / (k_.max() - k_.min() + 1e-9)); a.set_title(f"k{j}", fontsize=7); a.axis("off")
fig.suptitle(f"[EVIDENCIA] Los {w.shape[0]} kernels {w.shape[2]}x{w.shape[3]}x3 de la capa 0 (normalizados, RGB)")
guardar(fig, "3_kernels_capa0.png")

**¿Qué patrón aprendió cada convolución?** Para no atribuir significados sin comprobar, medimos dos cosas:

1. **Bordes:** correlación entre cada canal de la capa 0 y la magnitud del gradiente (Sobel) de la imagen.
   Una correlación alta comprueba que ese canal responde a cambios de intensidad (bordes). No dice nada de "frutas".
2. **Selectividad a las frutas:** activación media **dentro** de las máscaras anotadas ÷ activación media **fuera**
   (sin contar el relleno). Si el cociente crece con la profundidad, las capas profundas responden más a las
   regiones de fruta que al fondo en esta imagen. Es una medida, no una prueba de que un canal "detecte higos".

In [ ]:
# @title 3.3 Comprobaciones cuantitativas
gris = cv2.cvtColor(LB, cv2.COLOR_BGR2GRAY).astype(np.float32)
sobel = np.hypot(cv2.Sobel(gris, cv2.CV_32F, 1, 0), cv2.Sobel(gris, cv2.CV_32F, 0, 1))
a0 = ACT[0][0].float().cpu().numpy()
sob = cv2.resize(sobel, (a0.shape[2], a0.shape[1]), interpolation=cv2.INTER_AREA).ravel()
corr = [np.corrcoef(np.abs(a0[c]).ravel(), sob)[0, 1] for c in range(a0.shape[0])]
orden = np.argsort(corr)[::-1]
print("Capa 0: correlación |activación| vs. bordes Sobel por canal (mayor a menor):")
print("  " + ", ".join(f"c{c}={corr[c]:.2f}" for c in orden))

# Máscara de frutas y región válida en coordenadas del letterbox
fruta_lb = np.zeros(LB.shape[:2], bool)
fruta_lb[PADY:PADY + REDIM.shape[0], PADX:PADX + REDIM.shape[1]] = cv2.resize(
    GT_MASK.any(0).astype(np.uint8), (REDIM.shape[1], REDIM.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
valida = np.zeros(LB.shape[:2], bool)
valida[PADY:PADY + REDIM.shape[0], PADX:PADX + REDIM.shape[1]] = True
cocientes = []
for k in CAPAS_BB + [13, 16, 19, 22]:
    a = ACT[k][0].float().cpu().numpy()
    f_ = cv2.resize(fruta_lb.astype(np.uint8), (a.shape[2], a.shape[1]), interpolation=cv2.INTER_NEAREST).astype(bool)
    v_ = cv2.resize(valida.astype(np.uint8), (a.shape[2], a.shape[1]), interpolation=cv2.INTER_NEAREST).astype(bool)
    m = np.abs(a).mean(0)
    cocientes.append((k, a.shape, m[f_].mean() / (m[v_ & ~f_].mean() + 1e-9) if (v_ & ~f_).any() and f_.any() else np.nan))
print("\nActivación media dentro de frutas / fuera (fondo):")
for k, s, q in cocientes:
    print(f"  capa {k:>2} {type(red.model[k]).__name__:<6} {str(s[1:]):<15} cociente = {q:.2f}")

fig, axs = plt.subplots(1, 4, figsize=(18, 4))
axs[0].imshow(sobel, cmap="gray"); axs[0].set_title("Bordes (Sobel) de la entrada")
for a, c in zip(axs[1:3], orden[:2]):
    a.imshow(a0[c], cmap="viridis"); a.set_title(f"capa 0, canal {c}: corr = {corr[c]:.2f}")
axs[3].plot(range(len(cocientes)), [q for *_, q in cocientes], "o-")
axs[3].set_xticks(range(len(cocientes)), [str(k) for k, *_ in cocientes]); axs[3].axhline(1, ls="--", c="gray")
axs[3].set_xlabel("capa"); axs[3].set_title("activación fruta / fondo")
for a in axs[:3]: a.axis("off")
fig.suptitle("[EVIDENCIA] Comprobaciones: bordes en la capa 0 y selectividad por capa")
guardar(fig, "3_backbone_comprobaciones.png")

## 4. Integración de características (neck PAN-FPN)
- **Descendente (FPN):** el mapa más profundo (20×20, mucho contexto) se **sube** (`Upsample`, capas 11 y 14) y se
  **concatena** con mapas del backbone de más resolución (capas 6 y 4).
- **Ascendente (PAN):** se **baja** otra vez con convoluciones de stride 2 (capas 17 y 20) y se concatena (capas 18, 21).
- Resultado: tres mapas para la cabeza: **P3** (stride 8, frutas pequeñas), **P4** (stride 16) y **P5** (stride 32,
  frutas grandes). Cada uno combina detalle fino y contexto.

In [ ]:
# @title 4.1 Concatenaciones reales y mapas P3/P4/P5
print("Capas Concat: entradas -> salida (formas reales)")
for k, capa in enumerate(red.model):
    if type(capa).__name__ == "Concat":
        ent = ENTRADAS[k]
        print(f"  capa {k}: de {capa.f}: " + " + ".join(str(tuple(t.shape[1:])) for t in ent) + f"  ->  {tuple(ACT[k].shape[1:])}")
P_CAPAS = list(cabeza.f)
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for a, k, nom in zip(axs, P_CAPAS, ["P3", "P4", "P5"]):
    m = ACT[k][0].float().abs().mean(0).cpu().numpy()
    a.imshow(LB[..., ::-1]); a.imshow(cv2.resize(m, (LB.shape[1], LB.shape[0]), interpolation=cv2.INTER_NEAREST), alpha=0.6, cmap="jet")
    a.set_title(f"{nom} = capa {k}: {tuple(ACT[k].shape[1:])}, stride {LB.shape[0] // m.shape[0]}\n"
                f"cada celda ve {LB.shape[0] // m.shape[0]}x{LB.shape[0] // m.shape[0]} px de la entrada")
    a.axis("off")
fig.suptitle("[EVIDENCIA] Salidas del neck que recibe la cabeza (media de |activación|)")
guardar(fig, "4_neck_P3_P4_P5.png")

# [DIAGRAMA] con las formas reales
fig, ax = plt.subplots(figsize=(14, 6)); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
pos = {"entrada": (0.05, 0.5), 4: (0.28, 0.8), 6: (0.28, 0.5), 10: (0.28, 0.2),
       P_CAPAS[0]: (0.60, 0.8), P_CAPAS[1]: (0.60, 0.5), P_CAPAS[2]: (0.60, 0.2), "cabeza": (0.88, 0.5)}
for k, (x, y) in pos.items():
    if k == "entrada":
        txt, fc = f"entrada\n{tuple(X.shape[1:])}", "white"
    elif k == "cabeza":
        txt, fc = "cabeza Segment\ncajas + clases\n+ coeficientes\n+ Proto", "mistyrose"
    else:
        nom = {P_CAPAS[0]: "P3", P_CAPAS[1]: "P4", P_CAPAS[2]: "P5"}.get(k, "backbone")
        txt, fc = f"{nom} · capa {k}\n{tuple(ACT[k].shape[1:])}", "lightyellow" if k in P_CAPAS else "lightblue"
    ax.text(x, y, txt, fontsize=10, ha="center", va="center", bbox=dict(fc=fc, ec="k", boxstyle="round"))


def flecha(a_, b_, texto="", dx=0.075, curva=0.0, col="gray"):
    (x1, y1), (x2, y2) = pos[a_], pos[b_]
    if x1 == x2:   # vertical
        s_, e_ = (x1 + curva, y1 - 0.07 * np.sign(y1 - y2)), (x2 + curva, y2 + 0.07 * np.sign(y1 - y2))
    else:
        s_, e_ = (x1 + dx, y1), (x2 - dx, y2)
    ax.annotate("", e_, s_, arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
    if texto:
        if x1 == x2:
            ax.text(s_[0] + (0.01 if curva > 0 else -0.01), (s_[1] + e_[1]) / 2, texto, fontsize=8, color=col,
                    ha="left" if curva > 0 else "right", va="center")
        else:
            ax.text((s_[0] + e_[0]) / 2, (s_[1] + e_[1]) / 2 + 0.02, texto, fontsize=8, color=col, ha="center")


flecha("entrada", 6); flecha(4, 6); flecha(6, 10)
for b_, p_ in zip([4, 6, 10], P_CAPAS):
    flecha(b_, p_, "Concat lateral")
flecha(P_CAPAS[2], P_CAPAS[1], "Upsample (FPN)", curva=-0.05, col="tab:blue")
flecha(P_CAPAS[1], P_CAPAS[0], "Upsample (FPN)", curva=-0.05, col="tab:blue")
flecha(P_CAPAS[0], P_CAPAS[1], "Conv s2 (PAN)", curva=0.05, col="tab:green")
flecha(P_CAPAS[1], P_CAPAS[2], "Conv s2 (PAN)", curva=0.05, col="tab:green")
for p_ in P_CAPAS:
    flecha(p_, "cabeza")
ax.set_title("[DIAGRAMA] Backbone -> neck PAN-FPN -> cabeza (formas tomadas de los hooks; flujo simplificado)")
guardar(fig, "4_diagrama_arquitectura.png")

## 5. Cabeza de predicción
La cabeza `Segment` produce, para **cada celda de P3, P4 y P5** (80²+40²+20² = 8400 candidatos con entrada 640×640;
menos si hay relleno mínimo):

- **4 valores de caja** `cx, cy, w, h` en píxeles del tensor (ya decodificados desde la distribución DFL),
- **nc puntuaciones de clase** (sigmoide, ya en 0-1),
- **32 coeficientes de máscara**.

La rama **Proto** genera **32 prototipos** a 1/4 de resolución. La máscara de cada fruta es
$\sigma(\text{coef} \cdot \text{prototipos})$: una combinación lineal propia de cada candidato. Por eso dos frutas
de la misma clase salen con máscaras distintas.

In [ ]:
# @title 5.1 Forma del tensor de salida y ejemplos numéricos
def separar_salida(salida, nc, nm):
    # Busca en la salida (tuplas/dicts según la versión) el tensor (1, 4+nc+nm, N) y los prototipos (1, nm, h, w)
    preds = protos = None
    pila = [salida]
    while pila:
        o = pila.pop(0)
        if torch.is_tensor(o):
            if o.dim() == 3 and o.shape[1] == 4 + nc + nm and preds is None:
                preds = o
            elif o.dim() == 4 and o.shape[1] == nm and protos is None:
                protos = o
        elif isinstance(o, (list, tuple)):
            pila.extend(o)
        elif isinstance(o, dict):
            pila.extend(o.values())
    return preds, protos


PREDS, PROTOS = separar_salida(SALIDA, cabeza.nc, cabeza.nm)
P = PREDS[0].float()            # (4+nc+32, N)
PR = PROTOS[0].float()          # (32, mh, mw)
N = P.shape[1]
tam_escalas = [(X.shape[2] // int(s)) * (X.shape[3] // int(s)) for s in cabeza.stride]
inicio = np.cumsum([0] + tam_escalas)
print(f"Salida de la cabeza: {tuple(PREDS.shape)} = (lote, 4 caja + {cabeza.nc} clases + {cabeza.nm} coeficientes, {N} candidatos)")
print(f"Candidatos por escala: " + ", ".join(f"stride {int(s)}: {n}" for s, n in zip(cabeza.stride, tam_escalas)) + f"  (suma {sum(tam_escalas)})")
print(f"Prototipos: {tuple(PROTOS.shape)} (1/4 de la entrada {tuple(X.shape[2:])})")


def escala_de(i):
    e = int(np.searchsorted(inicio, i, side="right") - 1)
    s = int(cabeza.stride[e]); ancho = X.shape[3] // s
    j = i - inicio[e]
    return s, (j % ancho, j // ancho)


cajas_c, clases_c, coefs_c = P[:4].T, P[4:4 + cabeza.nc].T, P[4 + cabeza.nc:].T
mejores = clases_c.max(1).values.argsort(descending=True)[:5]
print("\nLos 5 candidatos con mayor puntuación (antes de umbrales y NMS):")
for i in mejores.tolist():
    s, (gx, gy) = escala_de(i)
    cx, cy, w_, h_ = cajas_c[i].tolist()
    print(f"  #{i:>5} stride {s:>2} celda ({gx},{gy})  caja cx={cx:.1f} cy={cy:.1f} w={w_:.1f} h={h_:.1f}  "
          f"clases={np.round(clases_c[i].cpu().numpy(), 3)}  coef[:6]={np.round(coefs_c[i, :6].cpu().numpy(), 2)}")

fig, axs = plt.subplots(4, 8, figsize=(16, 8))
for j, a in enumerate(axs.flat):
    a.imshow(PR[j].cpu(), cmap="RdBu_r"); a.set_title(f"proto {j}", fontsize=7); a.axis("off")
fig.suptitle(f"[EVIDENCIA] Los {PR.shape[0]} prototipos de máscara {tuple(PR.shape[1:])} de la rama Proto")
guardar(fig, "5_prototipos.png")

In [ ]:
# @title 5.2 De coeficientes a máscara para el mejor candidato (y la caja desde la DFL)
i = mejores[0].item()
logit = (coefs_c[i] @ PR.view(PR.shape[0], -1)).view(PR.shape[1:])
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
axs[0].bar(range(len(coefs_c[i])), coefs_c[i].cpu()); axs[0].set_title(f"32 coeficientes del candidato #{i}")
axs[1].imshow(logit.cpu(), cmap="RdBu_r"); axs[1].set_title("Σ coef_k · proto_k  (logits)")
axs[2].imshow(logit.sigmoid().cpu(), cmap="gray"); axs[2].set_title("sigmoide (probabilidad de fruta)")
axs[3].imshow(LB[..., ::-1]); cx, cy, w_, h_ = cajas_c[i].tolist()
axs[3].add_patch(plt.Rectangle((cx - w_ / 2, cy - h_ / 2), w_, h_, fill=False, ec="lime", lw=2))
axs[3].set_title(f"caja del candidato: {NOMBRES_MODELO[int(clases_c[i].argmax())]} {clases_c[i].max():.2f}")
for a in axs[1:]: a.axis("off")
fig.suptitle("[EVIDENCIA] La máscara de una instancia es una combinación lineal de los prototipos")
guardar(fig, "5_coeficientes_a_mascara.png")

# La caja viene de una distribución DFL de reg_max bins por lado (si esta versión expone la salida cruda)
crudo = SALIDA[1] if isinstance(SALIDA, (list, tuple)) and len(SALIDA) > 1 else None
if isinstance(crudo, dict) and "boxes" in crudo:
    s, (gx, gy) = escala_de(i)
    dist = crudo["boxes"][0, :, i].float().view(4, cabeza.reg_max).softmax(1).cpu()
    ltrb = (dist * torch.arange(cabeza.reg_max)).sum(1) * s
    ax_, ay_ = (gx + 0.5) * s, (gy + 0.5) * s
    print(f"DFL: distancias esperadas izq/arr/der/abajo = {ltrb.numpy().round(1)} px desde el centro de la celda ({ax_}, {ay_})")
    print(f"     caja reconstruida xyxy = ({ax_ - ltrb[0]:.1f}, {ay_ - ltrb[1]:.1f}, {ax_ + ltrb[2]:.1f}, {ay_ + ltrb[3]:.1f}); "
          f"caja de la cabeza = ({cx - w_ / 2:.1f}, {cy - h_ / 2:.1f}, {cx + w_ / 2:.1f}, {cy + h_ / 2:.1f})")
    fig, axs = plt.subplots(1, 4, figsize=(16, 2.5))
    for a, d, nom in zip(axs, dist, ["izquierda", "arriba", "derecha", "abajo"]):
        a.bar(range(cabeza.reg_max), d); a.set_title(f"DFL {nom}")
    fig.suptitle("[EVIDENCIA] Distribución de distancias (DFL) del mejor candidato")
    guardar(fig, "5_dfl.png")

## 6. Posprocesamiento
1. **Umbral de confianza** (`CONF`): se descartan candidatos cuya mejor clase no lo supera.
2. **Conversión** `cx, cy, w, h → x1, y1, x2, y2`.
3. **IoU** = área de intersección / área de unión entre dos cajas.
4. **NMS por clase** (`IOU_NMS`): se ordena por puntuación y se elimina todo candidato que se solape con uno mejor
   con IoU > umbral. Así se borran los **duplicados de la misma fruta**. Dos frutas que solo se tocan suelen tener IoU
   bajo y sobreviven. Si se superponen mucho, el NMS puede borrar una (error típico de este problema).
5. **Máscaras:** coeficientes × prototipos → sigmoide → recorte a la caja → subir a la resolución del tensor →
   quitar el relleno → redimensionar al tamaño original → umbral 0.5.

In [ ]:
# @title 6.1 Umbral, conversión de coordenadas, IoU y NMS
def xywh2xyxy(b):
    return torch.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)


def iou_manual(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter, union, inter / union


def postprocesar(P, conf=CONF, iou=IOU_NMS, nc=None):
    nc = nc or cabeza.nc
    puntaje, clase = P[4:4 + nc].max(0)
    idx = torch.nonzero(puntaje > conf).flatten()
    cajas = xywh2xyxy(P[:4, idx].T)
    keep = torchvision.ops.batched_nms(cajas, puntaje[idx], clase[idx], iou)
    return idx, cajas, puntaje[idx], clase[idx], keep


IDX, CAJAS, PUNT, CLS, KEEP = postprocesar(P)
if len(KEEP) == 0:
    print(f"Ningún candidato supera CONF={CONF}; para la demostración se baja a 0.05")
    IDX, CAJAS, PUNT, CLS, KEEP = postprocesar(P, conf=0.05)
print(f"Candidatos: {N} -> superan CONF={CONF}: {len(IDX)} -> tras NMS (IoU>{IOU_NMS}): {len(KEEP)} frutas")

# Demostración de IoU: (a) duplicado de la misma fruta, (b) dos frutas distintas que se tocan
k0 = KEEP[0].item()
ious = torchvision.ops.box_iou(CAJAS[k0:k0 + 1], CAJAS)[0]
ious[k0] = -1
dup = int(ious.argmax())
a, b = CAJAS[k0].tolist(), CAJAS[dup].tolist()
inter, union, v = iou_manual(a, b)
print(f"\n(a) Mejor candidato vs. su candidato más parecido: intersección={inter:.0f} px², unión={union:.0f} px², "
      f"IoU={v:.3f} (torchvision: {torchvision.ops.box_iou(CAJAS[[k0]], CAJAS[[dup]]).item():.3f}) "
      f"-> {'se ELIMINA (duplicado)' if v > IOU_NMS and CLS[dup] == CLS[k0] else 'se conserva'}")
if len(KEEP) > 1:
    ik = torchvision.ops.box_iou(CAJAS[KEEP[:1]], CAJAS[KEEP[1:]])[0]
    vec = KEEP[1:][ik.argmax()].item()
    inter, union, v = iou_manual(CAJAS[k0].tolist(), CAJAS[vec].tolist())
    print(f"(b) Dos frutas detectadas vecinas: intersección={inter:.0f} px², unión={union:.0f} px², IoU={v:.3f} "
          f"< {IOU_NMS} -> ambas se conservan")

fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for a_, cual, tit in [(axs[0], range(len(IDX)), f"Antes de NMS: {len(IDX)} candidatos > CONF"),
                      (axs[1], KEEP.tolist(), f"Después de NMS: {len(KEEP)} detecciones")]:
    a_.imshow(LB[..., ::-1])
    for n_, j in enumerate(cual):
        x1, y1, x2, y2 = CAJAS[j].tolist()
        a_.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color(int(CLS[j])) if a_ is axs[0] else color(n_), lw=0.7 if a_ is axs[0] else 2))
    a_.set_title(tit); a_.axis("off")
fig.suptitle("[EVIDENCIA] Selección de resultados con NMS (coordenadas del tensor)")
guardar(fig, "6_nms_antes_despues.png")

In [ ]:
# @title 6.2 Máscaras a resolución original: una máscara independiente por fruta
def recortar(m, cajas):
    n, h, w = m.shape
    x1, y1, x2, y2 = cajas[:, :, None, None].unbind(1)
    c = torch.arange(w, device=m.device)[None, None, :]
    r = torch.arange(h, device=m.device)[None, :, None]
    return m * ((c >= x1) & (c < x2) & (r >= y1) & (r < y2))


def mascaras_y_cajas(P, PR, idx, cajas_lb, keep, forma_lb, r, pad, forma0, umbral=UMBRAL_MASCARA, pasos=False):
    sel = idx[keep]
    coef = P[4 + cabeza.nc:, sel].T                               # (n, 32)
    c, mh, mw = PR.shape
    ih, iw = forma_lb
    logits = (coef @ PR.view(c, -1)).view(-1, mh, mw)            # (n, 160, 160)
    cajas_p = cajas_lb[keep] * torch.tensor([mw / iw, mh / ih, mw / iw, mh / ih], device=PR.device)
    m_proto = recortar(logits.sigmoid(), cajas_p)                # recorte a la caja (resolución de prototipos)
    m_lb = F.interpolate(m_proto[None], (ih, iw), mode="bilinear", align_corners=False)[0]  # a la resolución del tensor
    h0, w0 = forma0
    nh, nw = int(round(h0 * r)), int(round(w0 * r))
    m_sin = m_lb[:, pad[1]:pad[1] + nh, pad[0]:pad[0] + nw]       # quitar relleno
    m_0 = F.interpolate(m_sin[None], (h0, w0), mode="bilinear", align_corners=False)[0] > umbral  # a la original
    cajas0 = cajas_lb[keep].clone()
    cajas0[:, [0, 2]] = ((cajas0[:, [0, 2]] - pad[0]) / r).clamp(0, w0)   # recuperar coordenadas originales
    cajas0[:, [1, 3]] = ((cajas0[:, [1, 3]] - pad[1]) / r).clamp(0, h0)
    if pasos:
        return m_0, cajas0, (logits, m_proto, m_lb, m_sin)
    return m_0, cajas0


MASK0, CAJAS0, PASOS = mascaras_y_cajas(P, PR, IDX, CAJAS, KEEP, X.shape[2:], R, (PADX, PADY), (H0, W0), pasos=True)
MASK0 = MASK0.cpu().numpy(); CAJAS0 = CAJAS0.cpu().numpy()
PCLS = CLS[KEEP].cpu().numpy(); PSCORE = PUNT[KEEP].cpu().numpy()
print("Transformación de las máscaras (forma de cada paso):")
for nom, t in zip(["logits = coef·protos", "recortada a la caja", "subida al tensor", "sin relleno"], PASOS):
    print(f"  {nom:<22} {tuple(t.shape)}")
print(f"  {'resolución original':<22} {MASK0.shape}  (umbral {UMBRAL_MASCARA})")
print(f"Ejemplo de caja: tensor {CAJAS[KEEP[0]].cpu().numpy().round(1)} -> original {CAJAS0[0].round(1)}")

j = 0
fig, axs = plt.subplots(1, 5, figsize=(22, 4.5))
for a, t, tit in zip(axs[:4], PASOS, ["logits (resolución de prototipos)", "sigmoide recortada a la caja", "subida a la resolución del tensor", "sin relleno"]):
    a.imshow(t[j].float().cpu(), cmap="gray"); a.set_title(f"{tit}\n{tuple(t.shape[1:])}", fontsize=9)
axs[4].imshow(img0[..., ::-1]); axs[4].imshow(np.ma.masked_where(~MASK0[j], MASK0[j]), alpha=0.6, cmap="autumn")
axs[4].set_title(f"original {W0}x{H0}, umbral {UMBRAL_MASCARA}", fontsize=9)
for a in axs: a.axis("off")
fig.suptitle(f"[EVIDENCIA] Transformación de la máscara de la fruta 0 ({NOMBRES_MODELO[int(PCLS[j])]}) a la resolución original")
guardar(fig, "6_mascara_a_original.png")

n = len(MASK0)
cols = min(6, max(n, 1)); filas = int(np.ceil(n / cols))
fig, axs = plt.subplots(filas, cols, figsize=(3.2 * cols, 3 * filas), squeeze=False)
for j, a in enumerate(axs.flat):
    a.axis("off")
    if j < n:
        a.imshow(MASK0[j], cmap="gray")
        a.set_title(f"fruta {j}: {NOMBRES_MODELO[int(PCLS[j])]} {PSCORE[j]:.2f}\n{MASK0[j].sum():,} px", fontsize=8)
fig.suptitle("[EVIDENCIA] Una máscara binaria independiente por cada fruta detectada")
guardar(fig, "6_mascaras_independientes.png")

# Verificación contra el posprocesamiento interno de Ultralytics
if res_ultra.masks is not None:
    mu = F.interpolate(res_ultra.masks.data[None].float().cpu(), (H0, W0), mode="nearest")[0].numpy() > 0.5 \
        if res_ultra.masks.data.shape[1:] != (H0, W0) else res_ultra.masks.data.cpu().numpy() > 0.5
    print(f"\nUltralytics detecta {len(res_ultra.boxes)} frutas; nuestro posprocesamiento manual, {n}.")
    if len(res_ultra.boxes) == n and n:
        dif = np.abs(res_ultra.boxes.xyxy.cpu().numpy() - CAJAS0).max()
        print(f"Diferencia máxima entre cajas: {dif:.2f} px")

## 7. Resultado y evaluación
### 7.1 Visualización final y tiempo de procesamiento

In [ ]:
# @title 7.1 Resultado final y tiempos
fig, axs = plt.subplots(1, 2, figsize=(16, 7))
for a in axs: a.imshow(img0[..., ::-1]); a.axis("off")
for k, (c, poly) in enumerate(GT):
    axs[0].add_patch(plt.Polygon(poly, fc=(*color(k), 0.4), ec=color(k), lw=2))
capa_color = np.zeros((H0, W0, 4))
for j in range(len(MASK0)):
    capa_color[MASK0[j]] = (*color(j), 0.55)
    x1, y1, x2, y2 = CAJAS0[j]
    axs[1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color(j), lw=1.5))
    axs[1].text(x1, y1, f"{NOMBRES_MODELO[int(PCLS[j])]} {PSCORE[j]:.2f}", color="white", fontsize=8, bbox=dict(fc=color(j), ec="none", pad=1))
axs[1].imshow(capa_color)
axs[0].set_title(f"Anotación real: {len(GT)} frutas"); axs[1].set_title(f"Predicción: {len(MASK0)} frutas (máscara + caja)")
fig.suptitle("[EVIDENCIA] Imagen de seguimiento: resultado final")
guardar(fig, "7_resultado_final.png")


def sincronizar():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


tiempos = defaultdict(list)
for _ in range(20):
    sincronizar(); t0 = time.perf_counter()
    lb, _, r_, pad_ = letterbox(img0); x_ = a_tensor(lb).to(DEV)
    sincronizar(); t1 = time.perf_counter()
    with torch.no_grad():
        s_ = red(x_)
    sincronizar(); t2 = time.perf_counter()
    p_, pr_ = separar_salida(s_, cabeza.nc, cabeza.nm)
    out_ = postprocesar(p_[0].float())
    if len(out_[4]):
        mascaras_y_cajas(p_[0].float(), pr_[0].float(), *out_[:2], out_[4], x_.shape[2:], r_, pad_, (H0, W0))
    sincronizar(); t3 = time.perf_counter()
    for nom, v in zip(["preprocesamiento", "inferencia", "posprocesamiento"], [t1 - t0, t2 - t1, t3 - t2]):
        tiempos[nom].append(v * 1000)
print(f"Tiempo por imagen (mediana de 20 repeticiones, {'GPU' if torch.cuda.is_available() else 'CPU'}):")
for nom, v in tiempos.items():
    print(f"  {nom:<17} {np.median(v):7.1f} ms")
total = sum(np.median(v) for v in tiempos.values())
print(f"  {'total':<17} {total:7.1f} ms  (~{1000 / total:.0f} imágenes/s)")
print("Tiempos internos de Ultralytics en la predicción:", {k: round(v, 1) for k, v in res_ultra.speed.items()})

### 7.2 Métricas en el conjunto de prueba: mAP de máscaras e IoU
`val` empareja predicciones con anotaciones usando el **IoU de máscaras** con umbrales de 0.50 a 0.95.
**mAP50-95 (M)** es la métrica principal de segmentación de instancias. **(B)** es la de cajas.

In [ ]:
# @title 7.2 Validación en test
met = modelo.val(data=str(YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DISPOSITIVO, plots=True,
                 project=str(Path("runs_frutas").resolve()), name="test", exist_ok=True, verbose=False)
print(f"{'':<10}{'precisión':>10}{'recall':>9}{'mAP50':>9}{'mAP50-95':>10}")
for nom, m in [("Cajas (B)", met.box), ("Máscaras (M)", met.seg)]:
    print(f"{nom:<10}{m.mp:>10.3f}{m.mr:>9.3f}{m.map50:>9.3f}{m.map:>10.3f}")
print("\nPor clase (máscaras):")
print(f"{'clase':<14}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
for i, c in enumerate(met.ap_class_index):
    p_, r_, ap50, ap = met.seg.class_result(i)
    print(f"{NOMBRES_MODELO[int(c)]:<14}{p_:>7.3f}{r_:>7.3f}{ap50:>8.3f}{ap:>10.3f}")
print("Velocidad (ms/imagen):", {k: round(v, 1) for k, v in met.speed.items()})
for archivo in ["MaskPR_curve.png", "confusion_matrix_normalized.png"]:
    p_ = Path(met.save_dir) / archivo
    if p_.exists():
        fig = plt.figure(figsize=(9, 7)); plt.imshow(plt.imread(p_)); plt.axis("off"); plt.title(f"[EVIDENCIA] {archivo}")
        guardar(fig, f"7_{archivo}")

In [ ]:
# @title 7.3 IoU de máscaras en la imagen de seguimiento y análisis de aciertos y errores
def iou_mascaras(a, b):
    a = a.reshape(len(a), -1).astype(np.float32); b = b.reshape(len(b), -1).astype(np.float32)
    inter = a @ b.T
    return inter / (a.sum(1)[:, None] + b.sum(1)[None] - inter + 1e-9)


def emparejar(pm, pc, gm, gc, umbral=0.5):
    # Emparejamiento voraz por IoU de máscara. Devuelve TP [(p, g, iou)], FP [p], FN [g], errores de clase [(p, g)]
    if len(pm) == 0 or len(gm) == 0:
        return [], list(range(len(pm))), list(range(len(gm))), []
    M = iou_mascaras(pm, gm)
    tp, ec, usados_p, usados_g = [], [], set(), set()
    for p, g in sorted(np.argwhere(M >= umbral), key=lambda pg: -M[pg[0], pg[1]]):
        if p in usados_p or g in usados_g:
            continue
        usados_p.add(p); usados_g.add(g)
        (tp if pc[p] == gc[g] else ec).append((p, g, M[p, g]))
    fp = [p for p in range(len(pm)) if p not in usados_p]
    fn = [g for g in range(len(gm)) if g not in usados_g]
    return tp, fp, fn, ec


GT_CLS = np.array([c for c, _ in GT])
tp, fp, fn, ec = emparejar(MASK0, PCLS, GT_MASK, GT_CLS)
print(f"Imagen de seguimiento: {len(tp)} aciertos, {len(fp)} falsos positivos, {len(fn)} falsos negativos, {len(ec)} errores de clase")
for p, g, v in tp:
    print(f"  fruta real {g} ({NOMBRES_MODELO[int(GT_CLS[g])]}) <-> predicción {p}: IoU de máscara = {v:.3f}")
if tp:
    print(f"  IoU medio de máscaras: {np.mean([v for *_, v in tp]):.3f}")

# Recorrido por todo el conjunto de prueba
filas = []
toca_tot = toca_ok = sola_tot = sola_ok = 0
for res in modelo.predict([str(p) for p in IMAGENES["test"]], imgsz=IMGSZ, conf=CONF, iou=IOU_NMS, retina_masks=True,
                          stream=True, verbose=False, device=DISPOSITIVO):
    h, w = res.orig_shape
    anns = leer_etiquetas(Path(res.path), w, h)
    gm = mascaras_gt(anns, w, h); gc = np.array([c for c, _ in anns])
    pm = res.masks.data.cpu().numpy() > 0.5 if res.masks is not None else np.zeros((0, h, w), bool)
    pc = res.boxes.cls.cpu().numpy().astype(int) if res.boxes is not None else np.zeros(0, int)
    t_, f_, n_, e_ = emparejar(pm, pc, gm, gc)
    filas.append((res.path, len(t_), len(f_), len(n_), len(e_), len(anns)))
    tocan = {g for par in pares_que_se_tocan(gm) for g in par} if len(gm) > 1 else set()
    acertadas = {g for _, g, _ in t_}
    toca_tot += len(tocan); toca_ok += len(tocan & acertadas)
    sola_tot += len(gm) - len(tocan); sola_ok += len(acertadas - tocan)
TP, FP, FN, EC = (sum(f[i] for f in filas) for i in range(1, 5))
print(f"\nConjunto de prueba ({len(filas)} imágenes, IoU de máscara >= 0.5, CONF={CONF}):")
print(f"  aciertos={TP}  falsos positivos={FP}  falsos negativos={FN}  errores de clase={EC}")
print(f"  precisión={TP / max(TP + FP + EC, 1):.3f}  recall={TP / max(TP + FN + EC, 1):.3f}")
print(f"  recall en frutas que se TOCAN con otra: {toca_ok}/{toca_tot} = {toca_ok / max(toca_tot, 1):.3f}")
print(f"  recall en frutas AISLADAS:              {sola_ok}/{sola_tot} = {sola_ok / max(sola_tot, 1):.3f}")

peores = sorted(filas, key=lambda f: f[2] + f[3] + f[4], reverse=True)[:3]
mejores_img = sorted([f for f in filas if f[5] >= 3], key=lambda f: (f[2] + f[3] + f[4], -f[5]))[:1]
fig, axs = plt.subplots(1, 4, figsize=(24, 6))
for a, (ruta, nt, nf, nn, ne, ng) in zip(axs, mejores_img + peores):
    res = modelo.predict(ruta, imgsz=IMGSZ, conf=CONF, iou=IOU_NMS, retina_masks=True, verbose=False, device=DISPOSITIVO)[0]
    h, w = res.orig_shape
    anns = leer_etiquetas(Path(ruta), w, h); gm = mascaras_gt(anns, w, h); gc = np.array([c for c, _ in anns])
    pm = res.masks.data.cpu().numpy() > 0.5 if res.masks is not None else np.zeros((0, h, w), bool)
    pc = res.boxes.cls.cpu().numpy().astype(int) if res.boxes is not None else np.zeros(0, int)
    t_, f_, n_, e_ = emparejar(pm, pc, gm, gc)
    capa = np.zeros((h, w, 4))
    for p, _, _ in t_: capa[pm[p]] = (0, 1, 0, 0.45)
    for p in f_: capa[pm[p]] = (1, 0, 0, 0.5)
    for p, _, _ in e_: capa[pm[p]] = (1, 0, 1, 0.5)
    a.imshow(res.orig_img[..., ::-1]); a.imshow(capa)
    for g in n_:
        a.add_patch(plt.Polygon(anns[g][1], fill=False, ec="yellow", lw=2.5, ls="--"))
    a.set_title(f"{Path(ruta).name[:28]}\nTP={nt} FP={nf} FN={nn} clase={ne}", fontsize=9); a.axis("off")
fig.suptitle("[EVIDENCIA] Verde=acierto · rojo=falso positivo · magenta=clase equivocada · amarillo punteado=fruta no detectada\n"
             "(izquierda: un buen caso; derecha: los 3 casos con más errores)")
guardar(fig, "7_aciertos_y_errores.png")

### 7.4 Experimento: región de la caja vs. región de la máscara
Para cada fruta de la imagen de seguimiento se mide el área en píxeles de la **caja** y de la **máscara**, qué parte
de la caja es **fondo** y qué parte pertenece a **otras frutas**. Cuando dos frutas se tocan, la caja de una invade a
la otra; la máscara no.

In [ ]:
# @title 7.4 Área de caja vs. área de máscara
print(f"{'fruta':<7}{'clase':<14}{'área caja':>11}{'área máscara':>14}{'máscara/caja':>14}{'fondo en caja':>15}{'otras frutas en caja':>22}")
filas_exp = []
for j in range(len(MASK0)):
    x1, y1, x2, y2 = np.round(CAJAS0[j]).astype(int)
    caja = np.zeros((H0, W0), bool); caja[y1:y2, x1:x2] = True
    a_caja, a_mask = caja.sum(), (MASK0[j] & caja).sum()
    otras = (np.delete(MASK0, j, 0).any(0) & caja & ~MASK0[j]).sum() if len(MASK0) > 1 else 0
    fondo = a_caja - a_mask - otras
    filas_exp.append((a_caja, a_mask, fondo, otras))
    print(f"{j:<7}{NOMBRES_MODELO[int(PCLS[j])]:<14}{a_caja:>11,}{a_mask:>14,}{a_mask / max(a_caja, 1):>14.1%}"
          f"{fondo / max(a_caja, 1):>15.1%}{otras / max(a_caja, 1):>22.1%}")
if filas_exp:
    fe = np.array(filas_exp, dtype=float)
    print(f"\nPromedio: la máscara ocupa {np.mean(fe[:, 1] / fe[:, 0]):.1%} de su caja; "
          f"{np.mean(fe[:, 3] / fe[:, 0]):.1%} de cada caja pertenece a OTRAS frutas.")

    j = int(np.argmax(fe[:, 3])) if fe[:, 3].max() > 0 else 0
    x1, y1, x2, y2 = np.round(CAJAS0[j]).astype(int)
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    for a in axs: a.imshow(img0[..., ::-1]); a.axis("off")
    capa = np.zeros((H0, W0, 4)); capa[y1:y2, x1:x2] = (0, 0, 1, 0.4)
    axs[0].imshow(capa); axs[0].set_title(f"Caja de la fruta {j}: {int(fe[j, 0]):,} px")
    capa = np.zeros((H0, W0, 4)); capa[MASK0[j]] = (0, 1, 0, 0.5)
    axs[1].imshow(capa); axs[1].set_title(f"Máscara de la fruta {j}: {int(fe[j, 1]):,} px")
    capa = np.zeros((H0, W0, 4)); caja = np.zeros((H0, W0), bool); caja[y1:y2, x1:x2] = True
    capa[caja & ~MASK0[j]] = (1, 1, 0, 0.5)
    capa[caja & np.delete(MASK0, j, 0).any(0) & ~MASK0[j]] = (1, 0, 0, 0.6) if len(MASK0) > 1 else (1, 1, 0, 0.5)
    axs[2].imshow(capa); axs[2].set_title(f"Lo que la caja incluye de más: fondo (amarillo) {int(fe[j, 2]):,} px,\notras frutas (rojo) {int(fe[j, 3]):,} px")
    for a in axs: a.set_xlim(max(0, x1 - 40), min(W0, x2 + 40)); a.set_ylim(min(H0, y2 + 40), max(0, y1 - 40))
    fig.suptitle("[EVIDENCIA] Caja vs. máscara de la fruta que más invade a sus vecinas")
    guardar(fig, "7_caja_vs_mascara.png")

### Pregunta para defender: ¿por qué dos frutas de la misma clase necesitan máscaras diferentes?
Completen con sus resultados (secciones 5, 6 y 7):

- La **segmentación semántica** solo diría "estos píxeles son higo". Si dos higos se tocan, sus píxeles forman
  **una sola mancha** y no se puede contar ni medir cada uno.
- En YOLO-seg cada candidato tiene **sus propios 32 coeficientes** (sección 5). Con los mismos prototipos, cada fruta
  recibe una combinación diferente y además se **recorta a su propia caja**. Por eso salen máscaras independientes
  (sección 6.2), aunque la clase sea la misma.
- La caja no basta: en la sección 7.4 una caja incluye fondo y **parte de otras frutas**. Solo la máscara delimita el
  contorno real, y con ella se puede medir el área de cada fruta, contar o guiar a un robot.
- Limitación medida: comparen el **recall de frutas que se tocan vs. aisladas** (sección 7.3). Si es menor, el NMS (IoU
  de cajas) o la fusión de máscaras vecinas explican parte de los errores.

In [ ]:
# @title Descargar todas las figuras
shutil.make_archive("resultados_frutas", "zip", RES)
print("Figuras:", sorted(p.name for p in RES.iterdir()))
if EN_COLAB:
    from google.colab import files
    files.download("resultados_frutas.zip")